In [ ]:
!pip install -q transformers==4.45.0 trl==0.11.0 \
             datasets==2.19.0 accelerate==0.34.0 sentencepiece scipy lm-eval

import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 83.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 111.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [1]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # force single GPU


import torch
import torch.nn as nn
import gc, json, math
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer
from datasets import load_dataset

import lm_eval
from lm_eval.models.huggingface import HFLM

MODEL_ID    = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
OUTPUT_DIR  = "/kaggle/working/tinyllama-hmatrix-addon"
MAX_SEQ_LEN = 512
BATCH_SIZE  = 2
GRAD_ACCUM  = 16
LR          = 1e-4
EPOCHS      = 3
RANK        = 32
LEVEL   = 2
MIN_DIM     = 64
DROPOUT = 0.05
DEVICE      = "cuda:0"

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"rank={RANK}  min_dim={MIN_DIM}  → ~3.39M trainable params")
print(f"LoRA rank=32 reference → 3.60M trainable params")

2026-05-11 11:27:25.444732: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778498845.687658     128 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778498845.753468     128 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778498846.281321     128 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778498846.281365     128 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778498846.281367     128 computation_placer.cc:177] computation placer alr

GPU: Tesla T4
rank=32  min_dim=64  → ~3.39M trainable params
LoRA rank=32 reference → 3.60M trainable params


In [2]:
MAX_SEQ_LEN = 768

In [3]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------
# USV Block (leaf)
# -------------------------
class USVBlock(nn.Module):
    def __init__(self, out_dim, in_dim, rank, dropout=0.0):
        super().__init__()
        r = min(rank, min(out_dim, in_dim))

        # identical to lora_A
        self.V = nn.Parameter(torch.empty(r, in_dim))
        nn.init.kaiming_uniform_(self.V, a=math.sqrt(5))

        # identical to lora_B — zero init guarantees ΔW=0 at start
        self.U = nn.Parameter(torch.zeros(out_dim, r))

        self.dropout = nn.Dropout(p=dropout) if dropout > 0.0 else nn.Identity()

    def forward(self, x):
        return self.dropout(x) @ self.V.T @ self.U.T
# -------------------------
# Recursive H-Matrix Block
# -------------------------
class HMatrixBlock(nn.Module):
    def __init__(self, out_dim, in_dim, rank, level, min_dim=64, dropout=0.0):
        super().__init__()
        self.is_leaf = (level == 0 or min(out_dim, in_dim) <= min_dim)

        if self.is_leaf:
            self.block = USVBlock(out_dim, in_dim, rank, dropout)
            return

        self.in_mid  = in_dim  // 2
        self.out_mid = out_dim // 2

        self.A11 = HMatrixBlock(self.out_mid, self.in_mid, rank, level-1, min_dim, dropout)
        self.A22 = HMatrixBlock(self.out_mid, self.in_mid, rank, level-1, min_dim, dropout)
        self.A12 = USVBlock(self.out_mid, self.in_mid, rank, dropout)
        self.A21 = USVBlock(self.out_mid, self.in_mid, rank, dropout)

    def forward(self, x):
        if self.is_leaf:
            return self.block(x)

        x1 = x[..., :self.in_mid]
        x2 = x[..., self.in_mid:]

        y1 = self.A11(x1) + self.A12(x2)
        y2 = self.A21(x1) + self.A22(x2)

        return torch.cat([y1, y2], dim=-1)


# -------------------------
# H-Matrix Addon
# -------------------------
class HMatrixAddon(nn.Module):
    def __init__(self, out_dim, in_dim, rank, level=1, min_dim=64, dropout=0.0):
        super().__init__()
        self.hblock = HMatrixBlock(out_dim, in_dim, rank, level, min_dim, dropout)

    def forward(self, x):
        shape = x.shape
        return self.hblock(x.reshape(-1, shape[-1])).reshape(*shape[:-1], -1)


# -------------------------
# H-Matrix Linear (FINAL)
# -------------------------
class HMatrixLinear(nn.Module):
    def __init__(self, original_linear, rank, level=1, min_dim=64, dropout=0.0):
        super().__init__()

        out_dim, in_dim = original_linear.weight.shape
        dtype = original_linear.weight.dtype

        self.weight = nn.Parameter(
            original_linear.weight.data.clone(),
            requires_grad=False
        )
        self.bias = (
            nn.Parameter(original_linear.bias.data.clone(), requires_grad=False)
            if original_linear.bias is not None else None
        )

        self.h_addon = HMatrixAddon(out_dim, in_dim, rank, level, min_dim, dropout).to(dtype)

        self.rank  = rank
        self.alpha = 2 * rank
        self.scale = self.alpha / self.rank  # = 2

    def forward(self, x):
        base = F.linear(x, self.weight, self.bias)
        return base + self.scale * self.h_addon(x)


print("✅ H-matrix updated:")
print(" - kaiming_uniform V, zero-init S and U (matches LoRA init exactly)")
print(" - dropout threaded through all levels as param")
print(" - dropout=0.0 default for fair no-dropout comparison with LoRA")

✅ H-matrix updated:
 - kaiming_uniform V, zero-init S and U (matches LoRA init exactly)
 - dropout threaded through all levels as param
 - dropout=0.0 default for fair no-dropout comparison with LoRA


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map=DEVICE,
)

# Freeze all original params
for p in model.parameters():
    p.requires_grad = False

# Replace Q, K, V with HMatrixLinear (frozen original + trainable addon)
for i, layer in enumerate(model.model.layers):
    attn = layer.self_attn
    attn.q_proj = HMatrixLinear(attn.q_proj, RANK, LEVEL, MIN_DIM, dropout=DROPOUT).to(DEVICE)
    attn.k_proj = HMatrixLinear(attn.k_proj, RANK, LEVEL, MIN_DIM, dropout=DROPOUT).to(DEVICE)
    attn.v_proj = HMatrixLinear(attn.v_proj, RANK, LEVEL, MIN_DIM, dropout=DROPOUT).to(DEVICE)


trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} ({trainable/1e6:.2f}M)")
print(f"Total    : {total_p:,} ({total_p/1e6:.2f}M)")
print(f"\nLoRA reference: 3.60M trainable")
print(f"H-matrix addon: {trainable/1e6:.2f}M trainable")

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Trainable: 18,382,848 (18.38M)
Total    : 1,118,431,232 (1118.43M)

LoRA reference: 3.60M trainable
H-matrix addon: 18.38M trainable


In [16]:
from datasets import load_dataset

casehold = load_dataset("lex_glue", "case_hold")

casehold_train = casehold["train"]
casehold_val   = casehold["validation"]

print(casehold_train[0])

Generating train split:   0%|          | 0/45000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3600 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3900 [00:00<?, ? examples/s]

{'context': "Drapeau’s cohorts, the cohort would be a “victim” of making the bomb. Further, firebombs are inherently dangerous. There is no peaceful purpose for making a bomb. Felony offenses that involve explosives qualify as “violent crimes” for purposes of enhancing the sentences of career offenders. See 18 U.S.C. § 924(e)(2)(B)(ii) (defining a “violent felony” as: “any crime punishable by imprisonment for a term exceeding one year ... that ... involves use of explosives”). Courts have found possession of a'bomb to be a crime of violence based on the lack of a nonviolent purpose for a bomb and the fact that, by its very nature, there is a substantial risk that the bomb would be used against the person or property of another. See United States v. Newman, 125 F.3d 863 (10th Cir.1997) (unpublished) (<HOLDING>); United States v. Dodge, 846 F.Supp. 181,", 'endings': ['holding that possession of a pipe bomb is a crime of violence for purposes of 18 usc  3142f1', 'holding that bank robbery

In [17]:
casehold_train = casehold_train.shuffle(seed=42).select(range(5000))
casehold_val   = casehold_val.shuffle(seed=42).select(range(1000))

In [18]:
def format_casehold(ex):

    choices = "\n".join([
        f"({i}) {choice}"
        for i, choice in enumerate(ex["endings"])
    ])

    prompt = (
        f"Legal Context:\n{ex['context']}\n\n"
        f"Possible Holdings:\n{choices}\n\n"
        f"Answer: ("
    )

    answer = f"{ex['label']})"

    return {
        "prompt": prompt,
        "answer": answer,
        "text": prompt + answer
    }

In [19]:
casehold_train = casehold_train.map(
    format_casehold,
    remove_columns=casehold_train.column_names
)

casehold_val = casehold_val.map(
    format_casehold,
    remove_columns=casehold_val.column_names
)

print(casehold_train[0]["text"])

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Legal Context:
that could not be reconstructed and was, therefore, so deficient as to have made a fair and complete direct appeal of Perryman’s convictions an impossibility. The portions of the record cited to by Perryman at the hearing on his motion to reconstruct the record contain such statements as “Sidebar; inaudible to report.” See 2006 Transcript at 197, 200, 204, 207, 214, 225, 238. The transcript on the direct appeal from the 2006 trial included these notations indicating that certain portions were inaudible. Perryman did not raise the claims of an insufficient record on direct appeal and does not allege that his trial counsel or appellate counsel were ineffective on these bases. Consequently, we conclude that Perryman waived these claims. See Reed v. State, 866 N.E.2d 767, 768 (Ind.2007) (<HOLDING>); Sanders v. State, 765 N.E.2d 591, 592

Possible Holdings:
(0) holding that issues not properly presented to the bankruptcy court  cannot be raised  for the first time on appeal
(

In [20]:
def make_collator(tokenizer, max_len):
    def collate(batch):
        full_texts = [b["text"] for b in batch]
        
        enc = tokenizer(full_texts, truncation=True, max_length=max_len,
                        padding="max_length", return_tensors="pt")
        
        labels = enc["input_ids"].clone()

        for i, text in enumerate(full_texts):
            split_marker = "Answer: ("
            prompt_part  = text[:text.rfind(split_marker) + len(split_marker)]
            prompt_ids   = tokenizer(prompt_part, add_special_tokens=False)["input_ids"]
            prompt_len   = len(prompt_ids)
            labels[i, :prompt_len] = -100

        labels[labels == tokenizer.pad_token_id] = -100

        enc["labels"] = labels
        return enc
    return collate

In [21]:
data_collator=make_collator(tokenizer, MAX_SEQ_LEN)

In [22]:
from transformers import Trainer

model.enable_input_require_grads()

training_args = TrainingArguments(
    save_total_limit=1,
    output_dir="./output",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    fp16=False,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    group_by_length=False,
    optim="adamw_torch",
    weight_decay=0.01,
    remove_unused_columns=False,  # keep prompt/answer/text columns for collator
    load_best_model_at_end=False,
)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=make_collator(tokenizer, MAX_SEQ_LEN),
    train_dataset=casehold_train,
    eval_dataset=casehold_val,
    args=training_args,
)

trainer.train()
print("Done!")

Step,Training Loss,Validation Loss
100,0.522800,0.467413
200,0.250700,0.259490
300,0.216600,0.230092
400,0.149200,0.224762


Done!


In [27]:
from datasets import load_dataset

casehold_test = load_dataset(
    "lex_glue",
    "case_hold",
    split="test"
)

# Optional smaller subset for quick testing
casehold_test = (
    casehold_test
    .shuffle(seed=42)
    .select(range(1000))
)

print(casehold_test[0])

{'context': 'not be sued without consent in its own courts . . . .”). 8 Dempsey v. Bd. of Regents of the Univ. Sys. of Ga., 256 Ga. App. 291, 292 (568 SE2d 154) (2002) (punctuation omitted). 9 It appears from the correspondence that Shuford mailed a check to Walker in November 2009 to cover the repair and diminished value of Walker’s motorcycle. We reject Walker’s argument that this in any way waived sovereign immunity or estopped the State from invoking same because “a government official may not waive or be estopped from invoking statutory notice requirements.” Dempsey, 256 Ga. App. at 294; see also State of Ga. v. Haynes, 285 Ga. App. 637, 640 (647 SE2d 331) (2007) (“[T]he government may not waive or be estopped from invoking statutory notice requirements.”). 10 See Welch, 276 Ga. App. at 665 (<HOLDING>); Shelnutt v. Ga. Dep’t of Transp., 272 Ga.', 'endings': ['holding that because the plaintiffs did not strictly comply with the statutory provisions for sending ante litem notice the

In [28]:
import torch
from tqdm import tqdm

model.eval()

correct = 0
total = 0

for ex in tqdm(casehold_test):

    choices = ex["endings"]

    prompt = (
        f"Legal Context:\n{ex['context']}\n\n"
        f"Possible Holdings:\n"
    )

    for i, c in enumerate(choices):
        prompt += f"({i}) {c}\n"

    prompt += "\nAnswer: ("

    scores = []

    for i in range(5):

        candidate = f"{i})"

        full_text = prompt + candidate

        inputs = tokenizer(
            full_text,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SEQ_LEN
        ).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits[:, :-1]
        labels = inputs["input_ids"][:, 1:]

        loss_fct = torch.nn.CrossEntropyLoss(
            reduction="mean"
        )

        loss = loss_fct(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1)
        )

        scores.append(-loss.item())

    pred = scores.index(max(scores))

    if pred == ex["label"]:
        correct += 1

    total += 1


accuracy = 100 * correct / total

print(f"\nCaseHOLD TEST Accuracy: {accuracy:.2f}%")

100%|██████████| 1000/1000 [46:47<00:00,  2.81s/it]


CaseHOLD TEST Accuracy: 76.00%


In [ ]:
print(results_after)